# 05 — Hardware-Aware Compilation and Noise-Aware Metrics

This notebook demonstrates compiling the same circuit for different hardware targets and computing noise-aware metrics using the BackendV2 API.

**Key idea:** The same algorithm compiled for different hardware topologies produces different circuits with different expected fidelity. WestQuant Open's `noise_aware_metrics()` supports both BackendV1 and BackendV2 APIs.

In [ ]:
# Install if needed
# !pip install westquant[qiskit] pandas matplotlib

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.providers.fake_provider import GenericBackendV2
from westquant_qiskit import circuit_metrics, noise_aware_metrics
import pandas as pd
import matplotlib.pyplot as plt

print("WestQuant Open — Hardware-Aware Compilation")
print("=" * 50)

## Step 1: Build the Input Circuit

In [ ]:
# Build a QFT-6 circuit
n = 6
qc = QuantumCircuit(n)
qc.append(QFTGate(n), range(n))
qc = qc.decompose(reps=3)

m = circuit_metrics(qc)
print(f"Input: QFT-{n}")
print(f"  Depth:     {m['depth']}")
print(f"  2Q gates:  {m['two_qubit_gates']}")
print(f"  Size:      {m['size']}")

## Step 2: Compile for Different Hardware Targets

We use Qiskit's `GenericBackendV2` (the current BackendV2 API) to simulate different hardware topologies with noise models.

In [ ]:
# Create different BackendV2 backends with noise models
# GenericBackendV2 provides realistic noise via the Target API
backends = {
    "6q-noise": GenericBackendV2(num_qubits=6, noise_info=True, seed=42),
    "8q-noise": GenericBackendV2(num_qubits=8, noise_info=True, seed=43),
    "10q-noise": GenericBackendV2(num_qubits=10, noise_info=True, seed=44),
}

print("=== Hardware Targets (BackendV2) ===")
for name, backend in backends.items():
    print(f"\n  {name}:")
    print(f"    Backend: {backend.name}")
    print(f"    Num qubits: {backend.num_qubits}")
    print(f"    Has target: {hasattr(backend, 'target')}")
    print(f"    Target instructions: {len(backend.target.instructions)}")
    # Check that noise data is available via the V2 API
    for i, (instr, qargs) in enumerate(backend.target.instructions[:3]):
        ip = backend.target.instruction_properties(i)
        if ip:
            print(f"    {instr.name} on {qargs}: error={ip.error:.6f}, duration={ip.duration:.2e}")

## Step 3: Compile and Compute Noise-Aware Metrics

This is where the BackendV2 fix matters. `noise_aware_metrics()` now correctly reads gate error and duration data from the Target API.

In [ ]:
# Compile the same circuit for each hardware target and compute noise-aware metrics
results = []

for name, backend in backends.items():
    # Transpile for this backend
    compiled = transpile(qc, backend=backend, optimization_level=2, seed_transpiler=42)
    m = circuit_metrics(compiled)

    # Compute noise-aware metrics using the BackendV2 API
    # This is the function that was fixed to support Target-based backends
    noise = noise_aware_metrics(compiled, backend=backend)

    results.append({
        "backend": name,
        "num_qubits": backend.num_qubits,
        "depth": m["depth"],
        "two_qubit_gates": m["two_qubit_gates"],
        "size": m["size"],
        "swap_gates": m["swap_gates"],
        "estimated_fidelity": noise["estimated_fidelity"],
        "total_duration": noise["total_duration"],
        "noise_aware_depth": noise["noise_aware_depth"],
    })

    print(f"\n=== {name} ({backend.num_qubits}q) ===")
    print(f"  Depth:           {m['depth']}")
    print(f"  2Q gates:        {m['two_qubit_gates']}")
    print(f"  Size:            {m['size']}")
    print(f"  Est. fidelity:   {noise['estimated_fidelity']:.6f}" if noise['estimated_fidelity'] else "  Est. fidelity:   N/A")
    print(f"  Total duration:  {noise['total_duration']:.2e}s" if noise['total_duration'] else "  Total duration:  N/A")
    print(f"  Noise-aware depth: {noise['noise_aware_depth']:.4f}" if noise['noise_aware_depth'] else "  Noise-aware depth: N/A")

df = pd.DataFrame(results)
print("\n=== Hardware Comparison Table ===")
print(df.to_string(index=False))

In [ ]:
# Visualize the hardware comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Depth by backend
axes[0].bar(df["backend"], df["depth"], color="steelblue", edgecolor="black")
axes[0].set_ylabel("Depth")
axes[0].set_title("Circuit Depth by Hardware Target")
axes[0].tick_params(axis='x', rotation=45)

# 2Q gates by backend
axes[1].bar(df["backend"], df["two_qubit_gates"], color="coral", edgecolor="black")
axes[1].set_ylabel("2-Qubit Gates")
axes[1].set_title("2Q Gate Count by Hardware Target")
axes[1].tick_params(axis='x', rotation=45)

# Estimated fidelity by backend
fidelity_vals = [f if f is not None else 0 for f in df["estimated_fidelity"]]
axes[2].bar(df["backend"], fidelity_vals, color="green", edgecolor="black")
axes[2].set_ylabel("Estimated Fidelity")
axes[2].set_title("Estimated Fidelity by Hardware Target")
axes[2].set_ylim([min(fidelity_vals) * 0.9, 1.01])
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle(f"QFT-{n}: Same Circuit, Different Hardware Targets", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("hardware_targeting.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4: QPU Budget Prediction

WestQuant's QPU Budget Predictor estimates the required QPU budget from problem structure — before any quantum execution. This is the integration from Paper A.

In [ ]:
from qcsc import QPUBudgetPredictor, ProblemProfile

# Predict QPU budget for a QAOA MaxCut problem on different graph families
predictor = QPUBudgetPredictor()

print("=== QPU Budget Prediction ===")
print("(Calibrated from 12,600 exact simulation results)\n")

for family in ["ER", "BA", "WS", "GEO", "CONFIG"]:
    profile = ProblemProfile(
        problem="MaxCut",
        n_qubits=16,
        p=3,
        graph_family=family,
        quality_target=0.95,
    )
    budget = predictor.estimate(profile)
    print(f"  {family:8s}: {budget.recommended_shots:>6,} shots, "
          f"{budget.reduction_vs_naive:>6.0f}x reduction, "
          f"quality={budget.estimated_quality:.4f}")

print("\n=== Policy Comparison (MaxCut, n=16, p=3, GEO) ===")
profile = ProblemProfile(problem="MaxCut", n_qubits=16, p=3, graph_family="GEO")
for name, b in predictor.compare_policies(profile).items():
    print(f"  {name:20s}: {b.recommended_shots:>10,} shots, quality={b.estimated_quality:.4f}")

## Summary

| What | How |
|------|-----|
| Same circuit, different hardware | `transpile(qc, backend=backend)` |
| Noise-aware metrics (BackendV2) | `noise_aware_metrics(qc, backend=backend)` |
| QPU budget prediction | `QPUBudgetPredictor().estimate(profile)` |

The `noise_aware_metrics()` function now correctly supports the BackendV2 Target API — reading `instruction_properties(index).error` and `.duration` instead of the deprecated `properties().gate_error()` method.

**This is Experiment #18 from the project plan:** Noise/hardware-aware optimization.